# Practical PyTorch · Part 3 — Companion Notebook

This goes with **"What a Model Really Is: A Read-Only Peek Inside the Black Box"**. Run the cells top to bottom (Shift+Enter).

We're only *looking* inside a model here — no training, no math. A model is a tree of layers, each holding a pile of learned numbers. Let's see that for real.

## 1. Load a real pretrained model

ResNet-18 is a small, famous image classifier. `weights=ResNet18_Weights.DEFAULT` asks torchvision for the best official pretrained weights. The first run downloads them; after that they're cached.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)
print("loaded:", type(model).__name__)

## 2. Read the model like a table of contents

`print(model)` shows the **structure** — the nested tree of layers — not the millions of numbers inside. Read the *shape* of it, not every line. Notice it's modules inside modules inside modules, ending in an `fc` (a `Linear`) that produces 1000 scores, one per ImageNet category.

In [ ]:
print(model)

## 3. Count the parameters

Parameters are the learned numbers living inside the layers — the weights. When people say a model "has N parameters," this is what they're counting. ResNet-18 has about 11.7 million (small, by today's standards).

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"{total:,} parameters")

## 4. Peek at where the numbers live

`named_parameters()` gives every weight tensor with its name and shape. We slice to the first 5 — there are hundreds. Notice the names mirror the tree from `print(model)`, and every parameter is just a tensor with a shape (exactly like Part 2).

In [ ]:
for name, p in list(model.named_parameters())[:5]:
    print(name, tuple(p.shape))

## 5. The common layers, in plain English

You'll meet these three constantly. No math needed — just a one-line picture:

- **`nn.Linear`** — a *weighted sum*: list of numbers in, weighted-and-added list out.
- **`nn.Conv2d`** — *slides a small filter over an image*, reporting where it found a pattern.
- **`nn.ReLU`** — *keep the positives*: negatives become zero, positives pass through.

## 6. Build a tiny model and run a forward pass

`nn.Sequential` just means "run these layers in order." This little net takes 10 numbers, squeezes to 5, keeps the positives, then squeezes to 2. It's a complete `nn.Module` — same mechanics as ResNet, just tiny.

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(10, 5),   # weighted sum: 10 in, 5 out
    nn.ReLU(),          # keep the positives
    nn.Linear(5, 2),    # weighted sum: 5 in, 2 out
)

print(net)

Now feed it one random input and watch data flow through. The leading `1` in `torch.rand(1, 10)` is the batch dimension — "a batch of one example." Calling `net(x)` *is* a forward pass: data goes in the top, comes out the bottom transformed.

In [ ]:
x = torch.rand(1, 10)   # one example, 10 features
out = net(x)

print("input shape: ", tuple(x.shape))
print("output shape:", tuple(out.shape))
print("output:", out)

The two output numbers are noise — nobody trained this net, so its weights are random. But the *mechanics* are identical to ResNet's. Next post we hand a real pretrained model an actual image and read back what it thinks it's seeing.

We only looked here, and looking was enough to make a model ordinary.